In [ ]:
from pathlib import Path
import pickle as pkl
import json as json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
path = os.path.abspath("/path/to/code/labelnoise-on-UQ")
os.environ["PYTHONPATH"] = path + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, path)
from src.Utils.vocabulary import (
    TRAIN, TEST, MULTI_TARGET, HARD_TARGET, INDEX, LABELS, UQ, PREDICTED, AU, EU, TU,
    index2target_PTB, index2multitarget_ESR
)

%load_ext autoreload
%autoreload 2

In [ ]:
# General constants
dataset_idx2class = {"MNIST": {k: f"Nb {k}" for k in range(10)}, "ESR": index2multitarget_ESR, "PTB": index2target_PTB}
dataset_colors = {"MNIST": "green", "ESR": "orange", "PTB": "blue"}
nr_colors = {0.: '#93c47dff', 0.2: '#ffd966ff', 0.4: '#e69138ff', 0.6:'#cc0000ff'}
target_names = {"MNIST": HARD_TARGET, "ESR 1DCNNTransf": MULTI_TARGET, "ESR 2DCNN": MULTI_TARGET, "PTB": HARD_TARGET}
target_K = {"MNIST": 10, "ESR 1DCNNTransf": 5, "ESR 2DCNN": 5, "PTB": 2}

axis_fontsize = 14
title_fontsize = 16
legend_fontsize = 12

UQ_COLORS = {TU: 'skyblue', AU: 'lightgreen', EU: 'mistyrose'}
UQ_NAMES = [TU, AU, EU]
UQ_NAMES_SHORT = {TU: 'TU', AU: 'AU', EU: 'EU'}
MODES = [TRAIN, TEST]

loss_viz_names = {"CE_nodropout": "CE", 
                  "CE_dropout": "CE + Dropout", 
                  "GCE_q0-4_nodropout": "GCE (q=0.4)", "GCE_q0-7_nodropout": "GCE (q=0.7)", "GCE_q1_nodropout": "GCE (q=1.0)", 
                  "GCE_q0-4_dropout": "GCE (q=0.4) + Dropout", "GCE_q0-7_dropout": "GCE (q=0.7) + Dropout", "GCE_q1_dropout": "GCE (q=1.0) + Dropout", 
                  "RCE_A-16_nodropout": "RCE (A=16)",
                 "balancedCE_dropout": "balanced CE + Dropout", "balancedCE_nodropout": "balanced CE", 
                 "balancedGCE_q0-4_nodropout": "balanced GCE (q=0.4)", "balancedGCE_q0-7_nodropout": "balanced GCE (q=0.7)", "balancedGCE_q1_nodropout": "balanced GCE (q=1.0)",
                 "balancedGCE_q0-4_dropout": "balanced GCE (q=0.4) + Dropout", "balancedGCE_q0-7_dropout": "balanced GCE (q=0.7) + Dropout", "balancedGCE_q1_dropout": "balanced GCE (q=1.0) + Dropout",
                 "balancedRCE_A-16_nodropout": "balanced RCE (A=16)",
            }

def aggregate_prediction(pred):
    # pred: (n_classes, n_inference)
    mean_logits = np.mean(pred, axis=1)  # shape (n_classes,)
    return np.argmax(mean_logits)        # final predicted class


In [ ]:
# Gather paths

exp_root_path = Path("/path/to/experiments/publi_UQandNoise/Exp1")
exp_summary = {
    "MNIST": ["CE_dropout", "CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-4_dropout"],
    "ESR 1DCNNTransf": ["balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-4_dropout"],
    "ESR 2DCNN": ["balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-4_dropout"],
    "PTB": ["CE_dropout", "CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-4_dropout"]
}
exp_roots = {
    "MNIST": ("MNIST", "Exp_ClassifMNIST_2DCNN_"),
    "ESR 1DCNNTransf": ("ESR_1DCNNTransf", "Exp_ClassifESRmulti_1DCNNTransf_"),
    "ESR 2DCNN": ("ESR_2DCNN", "Exp_ClassifESRmulti_2DCNN_"),
    "PTB": ("PTB", "Exp_ClassifPTBsub_1DCNNTransf_")
}
exp_paths = {
    d_name : {
        l_name : {
            nr : exp_root_path/f"{exp_roots[d_name][0]}"/f"{exp_roots[d_name][1]}{l_name}"/f"noise_rate_{str(nr).replace('.', '-')}" for nr in [0., 0.2, 0.4, 0.6]
            } for l_name in exp_summary[d_name]
        } for d_name in exp_summary.keys()
}

nb_runs_per_exp = 10
run_paths = {
    d: 
        {l: 
            {nr: [exp_paths[d][l][nr]/f"run-{i}" for i in range(nb_runs_per_exp) if (exp_paths[d][l][nr]/f"run-{i}").exists()]
             for nr in exp_paths[d][l].keys()
             } 
            for l in exp_paths[d].keys()} 
        for d in exp_paths.keys()
    }

# dataset_names = ["MNIST", "PTB"]
dataset_names = ["ESR 1DCNNTransf", "ESR 2DCNN"]
n_dataset = len(dataset_names)
nr_rates = [0., 0.2, 0.4, 0.6]
modes = [TRAIN, TEST]
n_nr = len(nr_rates)
n_modes = len(modes)

# Predictions and metric without several inference with dropout
pred_name = "predictions.pkl"
metric_name = "metric.json"

# Predictions and metric without several inference with dropout
pred_name_MCDO = "predictions_MCdropout-entropy_ni5.pkl"
metric_name_MCDO = "metric_MCdropout-entropy_ni5.json"

# included_losses = {"MNIST": ["CE_nodropout", "GCE_q0-4_nodropout", "CE_dropout", "GCE_q0-4_dropout"], 
#                    "ESR 1DCNNTransf": ["balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedCE_dropout", "balancedGCE_q0-4_dropout"],
#                    "ESR 2DCNN": ["balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedCE_dropout", "balancedGCE_q0-4_dropout"],
#                    "PTB": ["CE_nodropout", "GCE_q0-4_nodropout", "CE_dropout", "GCE_q0-4_dropout"]}

included_losses = {"MNIST": ["CE_nodropout", "CE_dropout"], 
                   "ESR 1DCNNTransf": ["balancedCE_nodropout", "balancedCE_dropout"],
                   "ESR 2DCNN": ["balancedCE_nodropout", "balancedCE_dropout"],
                   "PTB": ["CE_nodropout", "CE_dropout"]}

svg_suffix = "_simple"

In [ ]:
# Print number of runs per dataset, loss and noise rate
for d_name in dataset_names:
    for l_name in exp_paths[d_name].keys():
        for nr in exp_paths[d_name][l_name].keys():
            print(f"{d_name} - {l_name} - nr {nr}: {len(run_paths[d_name][l_name][nr])} runs")

# Load predictions and metrics

In [ ]:
# Load predictions
predictions = {}
for d in dataset_names:
    predictions[d] = {}
    for l in exp_paths[d].keys():
        predictions[d][l] = {}
        for nr in nr_rates:
            predictions[d][l][nr] = []
            target_key = target_names[d]
            for run_i in run_paths[d][l][nr]:
                with (run_i/pred_name).open("rb") as f:
                    pred = pkl.load(f)
                predictions[d][l][nr].append(pred)


# <10s for 1 exp with and without dropout

In [ ]:
# Load predictions MCDO
predictions_MCDO = {}
for d in dataset_names:
    predictions_MCDO[d] = {}
    for l in exp_paths[d].keys():
        predictions_MCDO[d][l] = {}
        for nr in nr_rates:
            predictions_MCDO[d][l][nr] = []
            target_key = target_names[d]
            for run_i in run_paths[d][l][nr]:
                with (run_i/pred_name_MCDO).open("rb") as f:
                    pred = pkl.load(f)
                predictions_MCDO[d][l][nr].append(pred)

# <10s for 1 exp with and without dropout

In [ ]:
# Load metrics
# Dict description: metrics[dataset_name][noise_rate][loss_name] = list of metrics for each run
metrics = {}
for d in dataset_names:
    metrics[d] = {}
    for nr in nr_rates:
        metrics[d][nr] = {}
        for l in run_paths[d].keys():
            target_key = target_names[d]
            metrics[d][nr][l] = []
            for run_i in run_paths[d][l][nr]:
                with (run_i/metric_name).open("r") as f:
                    metric = json.load(f)
                metrics[d][nr][l].append(metric)

# Average metrics over runs
# Dict description: avg_metrics[dataset_name][noise_rate][loss_name] = dict of average metrics over runs
avg_metrics = {}
std_metrics = {}
for d in dataset_names:
    avg_metrics[d] = {}
    std_metrics[d] = {}
    for nr in nr_rates:
        avg_metrics[d][nr] = {}
        std_metrics[d][nr] = {}
        for l in run_paths[d].keys():
            target_key = target_names[d]
            # Average metrics over runs
            avg_metrics[d][nr][l] = {}
            std_metrics[d][nr][l] = {}
            print(f"number of runs for {d} - {l} - nr {nr}: {len(metrics[d][nr][l])}")
            for mode in metrics[d][nr][l][0].keys():
                avg_metrics[d][nr][l][mode] = {}
                std_metrics[d][nr][l][mode] = {}
                for m in metrics[d][nr][l][0][mode].keys():
                    if type(metrics[d][nr][l][0][mode][m]) in [int, float]:
                        print(m, "int/float", mode)
                        avg_metrics[d][nr][l][mode][m] = np.mean([metrics[d][nr][l][i][mode][m] for i in range(len(metrics[d][nr][l]))])
                        std_metrics[d][nr][l][mode][m] = np.std([metrics[d][nr][l][i][mode][m] for i in range(len(metrics[d][nr][l]))])
                    else:
                        print(m, "None/0", mode)
                        avg_metrics[d][nr][l][mode][m] = None
                        std_metrics[d][nr][l][mode][m] = None

In [ ]:
# Load metrics MCDO
# Dict description: metrics[dataset_name][noise_rate][loss_name] = list of metrics for each run
metrics_MCDO = {}
for d in dataset_names:
    metrics_MCDO[d] = {}
    for nr in nr_rates:
        metrics_MCDO[d][nr] = {}
        for l in run_paths[d].keys():
            target_key = target_names[d]
            metrics_MCDO[d][nr][l] = []
            for run_i in run_paths[d][l][nr]:
                with (run_i/metric_name).open("r") as f:
                    metric = json.load(f)
                metrics_MCDO[d][nr][l].append(metric)

# Average metrics over runs
# Dict description: avg_metrics[dataset_name][noise_rate][loss_name] = dict of average metrics over runs
avg_metrics_MCDO = {}
std_metrics_MCDO = {}
for d in dataset_names:
    avg_metrics_MCDO[d] = {}
    std_metrics_MCDO[d] = {}
    for nr in nr_rates:
        avg_metrics_MCDO[d][nr] = {}
        std_metrics_MCDO[d][nr] = {}
        for l in run_paths[d].keys():
            target_key = target_names[d]
            # Average metrics over runs
            avg_metrics_MCDO[d][nr][l] = {}
            std_metrics_MCDO[d][nr][l] = {}
            print(f"number of runs for {d} - {l} - nr {nr}: {len(metrics_MCDO[d][nr][l])}")
            for mode in metrics_MCDO[d][nr][l][0].keys():
                avg_metrics_MCDO[d][nr][l][mode] = {}
                std_metrics_MCDO[d][nr][l][mode] = {}
                for m in metrics_MCDO[d][nr][l][0][mode].keys():
                    if type(metrics_MCDO[d][nr][l][0][mode][m]) in [int, float]:
                        print(m, "int/float", mode)
                        avg_metrics_MCDO[d][nr][l][mode][m] = np.mean([metrics_MCDO[d][nr][l][i][mode][m] for i in range(len(metrics_MCDO[d][nr][l]))])
                        std_metrics_MCDO[d][nr][l][mode][m] = np.std([metrics_MCDO[d][nr][l][i][mode][m] for i in range(len(metrics_MCDO[d][nr][l]))])
                    else:
                        print(m, "None/0", mode)
                        avg_metrics_MCDO[d][nr][l][mode][m] = None
                        std_metrics_MCDO[d][nr][l][mode][m] = None

# Theoretical functions

In [ ]:
# Theoretical minimal top score for each loss and each noise rate (for a given K)
from pathlib import Path


def v_min_CE(eta, K) -> float or np.ndarray:
    return 1 - eta

def v_min_MAE(eta, K) -> float or np.ndarray:
    return 1 *(eta <= (K - 1)/K) # if eta <= (K - 1)/K, v_min = 0, else v_min = 1 (boolean to keep vectorization)

def v_min_GCE(eta, K, q) -> float or np.ndarray:
    if q == 1:
        return 1
    return 1 / (1 + (K - 1) ** (-q / (1 - q)) * (eta / (1 - eta)) ** (1 / (1 - q)))

    
def loss_theo_min(loss_name):
    if "GCE" in loss_name:
        if "q0-4" in loss_name:
            return lambda eta, K: v_min_GCE(eta, K, q=0.4)
        elif "q0-7" in loss_name:
            return lambda eta, K: v_min_GCE(eta, K, q=0.7)
    # if "balanced_geoJSD" in loss_name:
    #     return lambda eta, K: None
    elif "MAE" in loss_name:
        return v_min_MAE
    elif "CE" in loss_name:
        return v_min_CE
    print(f"No theoretical minimum defined for loss {loss_name}")
    return lambda eta, K: None


# Scores at the end of training - Mean of max scores

In [ ]:
mode = TRAIN

# Get run top scores
byrun_scores_ordered = {}
for d_name in dataset_names:
    loss_names = included_losses[d_name]
    byrun_scores_ordered[d_name] = {}
    # Get the scores list for each loss and each nr
    for i_l, l_name in enumerate(loss_names):
        byrun_scores_ordered[d_name][l_name] = {}
        for nr in nr_rates:
            byrun_scores_ordered[d_name][l_name][nr] = []
            predictions_nr_l = predictions[d_name][l_name][nr]
            for i_run in range(len(predictions_nr_l)):
                # Initial shape is (n_samples, n_classes, n_inference), with n_inference = 1 in our case
                # Sort the predictions (and labels and index for coherence) according to index to match between runs
                sample_idx = predictions_nr_l[i_run][mode][INDEX]
                predictions_nr_l[i_run][mode][INDEX] = predictions_nr_l[i_run][mode][INDEX][np.argsort(sample_idx)]
                predictions_nr_l[i_run][mode][target_names[d_name]][LABELS] = predictions_nr_l[i_run][mode][target_names[d_name]][LABELS][np.argsort(sample_idx)]
                predictions_nr_l[i_run][mode][target_names[d_name]][PREDICTED] = predictions_nr_l[i_run][mode][target_names[d_name]][PREDICTED][np.argsort(sample_idx)]
                
                # Mean across the different inferences (final shape (n_samples, n_classes))
                run_top_scores = np.mean(predictions_nr_l[i_run][mode][target_names[d_name]][PREDICTED], axis=2)

                byrun_scores_ordered[d_name][l_name][nr].append(run_top_scores)  # shape (n_samples,)


In [ ]:
# Mean of max scores
top_scores_mean = {}
for d_name in dataset_names:
    loss_names = included_losses[d_name]
    top_scores_mean[d_name] = {}
    
    # Get the scores list for each loss and each nr
    for i_l, l_name in enumerate(loss_names):
        top_scores_mean[d_name][l_name] = {}
        for nr in nr_rates:
            top_scores_mean[d_name][l_name][nr] = []
            for i_run in range(len(predictions[d_name][l_name][nr])):
                # Initial shape is (n_samples, n_classes)
                byrun_score = byrun_scores_ordered[d_name][l_name][nr][i_run]  # shape (n_samples, n_classes)
                # Top score across the classes (final shape (n_samples,))
                run_top_scores = np.max(byrun_score, axis=1)  # shape (n_samples,)
                top_scores_mean[d_name][l_name][nr].append(run_top_scores)
            # Mean across the runs (final shape (n_samples,)), average can be done because samples are sorted according to index and thus correspond between runs
            top_scores_mean[d_name][l_name][nr] = np.mean(top_scores_mean[d_name][l_name][nr], axis=0)

In [ ]:
score_names = {"Mean of max scores": top_scores_mean}

In [ ]:
def compare_value_to_bin(value, bin_edges):
    """
    Compare a value to a couple of bin edges and return the distance between the value and the closest bin edge.
    
    Parameters:
    - value: The value to compare.
    - bin_edges: A pair of bin edges.
    
    Returns:
    - The distance between the value and the closest bin edge.
    """
    lower_edge, upper_edge = bin_edges
    if value < lower_edge:
        distance = value - lower_edge
    elif value > upper_edge:
        distance = value - upper_edge
    else:
        distance = 0  # The value is within the bin edges
    return distance

def compare_dist_value_closest_global_max(exp_bin_values, test_theo, bin_edges=None):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    peak_values = np.where(exp_bin_values == np.max(exp_bin_values))[0] # indices of the global maximum values
    peak_edges = [(bin_edges[p], bin_edges[p + 1]) for p in peak_values] # corresponding bin edges for the global maximum values
    distance_to_edges = [compare_value_to_bin(test_theo, edges) for edges in peak_edges] # distances to the bin edges of the global maximum values
    index_of_closest_dist = np.argmin(np.abs(distance_to_edges)) # index of the closest global maximum to the theoretical value
    return distance_to_edges[index_of_closest_dist] / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

def get_closest_local_max(exp_bin_values, test_theo, bin_edges):
    assert bin_edges is not None, "bin_edges must be provided to get_closest_local_max function."
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    # --- Get all local maxima (including plateaus and border values) ---
    # Search for local maxima
    local_maxima = (np.diff(np.sign(np.diff(exp_bin_values))) < 0).nonzero()[0] + 1  # indices of local maxima
    
    # Get plateaus (consecutive indices with the same value) and add them to local maxima
    plateaus = np.where(np.diff(exp_bin_values) == 0)[0] + 1
    for p in plateaus:
        if (p - 1 in local_maxima or p + 1 in local_maxima) and p not in local_maxima:
            local_maxima = np.append(local_maxima, p)
    
    # Check if value at the borders are local maxima
    if exp_bin_values[-1] > exp_bin_values[-2] and (len(exp_bin_values) - 1) not in local_maxima:
        local_maxima = np.append(local_maxima, len(exp_bin_values) - 1)
    if exp_bin_values[0] > exp_bin_values[1] and 0 not in local_maxima:
        local_maxima = np.append(local_maxima, 0)
    
    if len(local_maxima) == 0:
        return None, None, None  # No local maxima found
    
    # Get the corresponding bin edges for the local maxima and distance to theoretical value
    local_maxima_edges = [(bin_edges[l], bin_edges[l + 1]) for l in local_maxima]
    distance_to_edges = [compare_value_to_bin(test_theo, edges) for edges in local_maxima_edges] # distances to the bin edges of the local maximum values
    index_of_closest_dist = np.argmin(np.abs(distance_to_edges)) # index of the closest local maximum to the theoretical value
    
    closest_local_max = local_maxima[index_of_closest_dist]
    distance_to_closest_local_max = distance_to_edges[index_of_closest_dist]
    return closest_local_max, distance_to_closest_local_max, local_maxima

# Vizualize scores

In [ ]:
# Gather histograms
n_bins = 40
bin_edges = np.linspace(0, 1, n_bins + 1)  # n_bins bins from 0 to 1

hist_lists = {}
for d_name in dataset_names:
    print(d_name)
    for i_score, (score_name, exp_score) in enumerate(score_names.items()):
        exp_score_d = exp_score[d_name]
        
        hist_list = {}
        for l_name in exp_score_d.keys():
            hist_list[l_name] = {}
            for nr in exp_score_d[l_name].keys():
                exp_scores = exp_score_d[l_name][nr]
                hist, _ = np.histogram(exp_scores, bins=bin_edges)
                hist_list[l_name][nr] = hist
        
        # Gather histograms and scores
        hist_lists[(d_name, score_name)] = hist_list

In [ ]:
# Get theoretical minimum for each loss and each noise rate

theoretical_mins = {}
for d_name in dataset_names:
    theoretical_mins[d_name] = {}
    loss_names = included_losses[d_name]
    for i_l, l_name in enumerate(loss_names):
        theoretical_mins[d_name][l_name] = {}
        for nr in nr_rates:
            # Get the theoretical minimum for the loss and noise rate
            K = target_K[d_name]
            eta = nr
            v_min_fn = loss_theo_min(l_name)
            v_min = v_min_fn(eta, K)
            
            # Get the theoretical minimum bin index
            theo_min = loss_theo_min(l_name)(nr, target_K[d_name])
            theoretical_mins[d_name][l_name][nr] = theo_min
print("Theoretical minimums:", theoretical_mins)

In [ ]:
# Visualization with mean of max histograms, each column a loss function, each row a nr rate
print(score_names.keys())
score_name = "Mean of max scores" # 'Max score concat', 'Mean of max scores', 'Max of mean score'
print(f"Visualizing histograms for score type {score_name}")
individual_title_fontsize = 10

publi_mode=True
publi_colors = "gray"
publi_theo = ("red", "-")

for d_name in dataset_names:
    loss_names = included_losses[d_name]
    fig, axs = plt.subplots(len(nr_rates), len(included_losses[d_name]),  figsize=(4 * len(included_losses[d_name]), 2 * len(nr_rates)))
    for i_l, l_name in enumerate(loss_names):
        for i_nr, nr in enumerate(nr_rates):
            # Get histogram
            hist_values = hist_lists[(d_name, score_name)][l_name][nr]
            # Plot histogram
            bin_width = bin_edges[1] - bin_edges[0]
            axs[i_nr, included_losses[d_name].index(l_name)].bar(bin_edges[:-1] + bin_width / 2, hist_values, width=bin_width, color=nr_colors[nr] if not publi_mode else publi_colors, alpha=0.5)

            if publi_mode:
                axs[i_nr, included_losses[d_name].index(l_name)].set_xlabel("Score maximum", fontsize=axis_fontsize)
                axs[i_nr, included_losses[d_name].index(l_name)].set_ylabel("Count", fontsize=axis_fontsize)
            else:
                axs[i_nr, included_losses[d_name].index(l_name)].set_xlabel("Top score", fontsize=axis_fontsize)
                axs[i_nr, included_losses[d_name].index(l_name)].set_ylabel("Count", fontsize=axis_fontsize)
            axs[i_nr, included_losses[d_name].index(l_name)].set_xticks(np.linspace(0, 1, 5))
            axs[i_nr, included_losses[d_name].index(l_name)].set_ylim(0, max(hist_values) * 1.1)
            axs[i_nr, included_losses[d_name].index(l_name)].set_xlim(0, 1)

        
            # Quantify distance with theory
            theo_min = theoretical_mins[d_name][l_name][nr]
            if theo_min is not None:
                axs[i_nr, included_losses[d_name].index(l_name)].axvline([theo_min], color=nr_colors[nr] if not publi_mode else publi_theo[0], linestyle="--" if not publi_mode else publi_theo[1], label="Theoretical optimum")

            avg_metric_info = f"\nAvg MCC test: {avg_metrics[d_name][nr][l_name][mode]['mcc ' + target_names[d_name]]:.2f} ± {std_metrics[d_name][nr][l_name][mode]['mcc ' + target_names[d_name]]:.2f}" if avg_metrics[d_name][nr][l_name][mode]['mcc ' + target_names[d_name]] is not None else "N/A"
            loss_str = loss_viz_names.get(l_name, l_name)
            title_full_str = f"Noise rate {nr}, loss {loss_str}"
            axs[i_nr, included_losses[d_name].index(l_name)].set_title(title_full_str, fontsize=title_fontsize)

    axs[0, 1].set_xlim(0, 1)
    plt.suptitle(f"{d_name} - {mode}", fontsize=individual_title_fontsize)
    plt.tight_layout()
    # Save svg figure
    plt.savefig(exp_root_path/exp_roots[d_name][0]/f"histograms_{d_name.replace(' ', '')}_{score_name.replace(' ', '_')}_dropoutOrNot_nbins{n_bins}{svg_suffix}.svg", format='svg')
    plt.show()

# 8s

In [ ]:
# Visualization with matrix of bins
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.patches import Patch

publi_mode=True
publi_colors = "gray"
publi_theo_pt = ("red", "-")
publi_theo_line = ("black", "-")
publi_theo_pt_exp = ("orange", "-")
publi_orientation = "horizontal" # "horizontal" or "vertical"

score_name = "Mean of max scores" # 'Max score concat', 'Mean of max scores', 'Max of mean score'
for d_name in dataset_names:
    is_PTB = d_name == "PTB"
    loss_names = included_losses[d_name]
    fig, axs = plt.subplots(1, len(included_losses[d_name]) + 1, figsize=(3.5 * len(included_losses[d_name]), 3), gridspec_kw={'width_ratios': [1] * len(included_losses[d_name]) + [0.04]})
    # fig, axs = plt.subplots(1, len(included_losses[d_name]) + 1, figsize=(4 * len(included_losses[d_name]), 4), gridspec_kw={'width_ratios': [1] * len(included_losses[d_name]) + [0.04]})
    for i_l, l_name in enumerate(included_losses[d_name]):
        # Get histogram
        hist_values = hist_lists[(d_name, score_name)][l_name][nr]
        # Gather the bins of each noise rate for the current loss
        bins_matrix = np.array([hist_lists[(d_name, score_name)][l_name][nr] for nr in nr_rates if not(nr == 0.6 and is_PTB)])  # shape (n_nr, n_bins)
        print(bins_matrix.shape)
        bins_matrix = bins_matrix.T  # shape (n_bins, n_nr), and lower edges at the bottom of the heatmap

        # Show the bins matrix as a heatmap
        eps = 100
        sns.heatmap(
            # bins_matrix + eps,
            # norm=LogNorm(vmin=eps, vmax=bins_matrix.max() + eps),
            bins_matrix,
            annot=False,
            # cmap="crest",
            cmap="Greys",
            xticklabels=[nr for nr in nr_rates if not(nr == 0.6 and is_PTB)],
            ax=axs[i_l],
            cbar_kws={"label": "Count"},
            cbar=(i_l == len(included_losses[d_name]) - 1),
            cbar_ax=axs[-1]
        )
        step = 10
        
        # Show theoretical values on the heatmap
        # theo_indices = [np.searchsorted(bin_edges, theoretical_mins[d_name][l_name][nr]) - 1 for nr in nr_rates if not(nr == 0.6 and is_PTB)]
        theo_indices = [theoretical_mins[d_name][l_name][nr] * (len(bin_edges) - 1) for nr in nr_rates if not(nr == 0.6 and is_PTB)]
        sns.lineplot(
            x=[i + 0.5 for i in range(len(theo_indices))], 
            y=theo_indices, 
            color=publi_theo_line[0], 
            linestyle=publi_theo_line[1], 
            marker="o", 
            markersize=3,
            markeredgecolor=publi_theo_line[0],
            markerfacecolor=publi_theo_line[0],
            label="Theo. optimum trend", 
            ax=axs[i_l], 
            legend=False
            )
        sns.scatterplot(
            x=[i + 0.5 for i in range(len(theo_indices))], 
            y=theo_indices, 
            color=publi_theo_pt[0], marker="o", s=100, 
            label="Theo. optimum", 
            ax=axs[i_l], 
            legend=False
            )
        
        
        # Show closest local max values on the heatmap and annotate the distance between theoretical and closest local max
        local_maxs = []
        for i_nr, nr in enumerate(nr_rates):
            if not(nr == 0.6 and is_PTB):
                closest_local_max, _, _ = get_closest_local_max(exp_bin_values=hist_lists[(d_name, score_name)][l_name][nr], bin_edges=bin_edges, test_theo=theoretical_mins[d_name][l_name][nr])
                local_maxs.append(closest_local_max)
                # distance = distance_lists[(d_name, score_name)][l_name][nr]
                # axs[i_l].annotate(f"{distance:.2f}", xy=(i_nr + 0.5, closest_local_max), xytext=(i_nr + 0.5, closest_local_max + 2), ha='center', fontsize=8, color="blue")
        # -- with a line plot
        # sns.lineplot(x=[i + 0.5 for i in range(len(local_maxs))], y=local_maxs, linestyle="--", color=publi_theo_line[0], marker="o", markersize=5, label="Closest local maximum", ax=axs[i_l], legend=False)
        sns.lineplot(
            x=[i + 0.5 for i in range(len(local_maxs))], 
            y=[lm + 0.5 for lm in local_maxs], 
            linestyle="--", 
            marker="o", 
            markersize=3, 
            markeredgecolor=publi_theo_line[0],
            markerfacecolor=publi_theo_line[0],
            color=publi_theo_line[0], 
            label="Exp. closest local max. trend", 
            ax=axs[i_l], 
            legend=False)
                
        # -- with a matrix
        exp_matrix = np.zeros_like(bins_matrix)
        for i_nr, lm in enumerate(local_maxs):
            exp_matrix[lm, i_nr] = 1
        cmap = ListedColormap([
            (1, 1, 1, 0),          # 0 → transparent
            publi_theo_pt_exp[0]    # 1 → your color
        ])
        sns.heatmap(
            exp_matrix,
            annot=False,
            cmap=cmap,
            vmin=0,
            vmax=1,
            xticklabels=[nr for nr in nr_rates if not (nr == 0.6 and is_PTB)],
            ax=axs[i_l],
            cbar=False,
            alpha=0.5
        )
        axs[i_l].axes.invert_yaxis()  # Invert y-axis to have lower bins at the bottom
        axs[i_l].set_yticks(np.concatenate([np.arange(0, len(bin_edges) - 1, step), [len(bin_edges) - 1]]))
        axs[i_l].set_yticklabels(np.concatenate([np.round(bin_edges[:-1][::step], 2), [np.round(bin_edges[-1], 2)]]), rotation=0)
        axs[i_l].set_ylabel(score_name, fontsize=axis_fontsize)
        axs[i_l].set_xlabel("Noise rate", fontsize=axis_fontsize)
        axs[i_l].set_title(f"{loss_viz_names.get(l_name, l_name)}", fontsize=title_fontsize)
    # Put legend below the subplots and reserve enough space so it does not overlap the x-axis
    handles, labels = axs[i_l].get_legend_handles_labels()
    handles.append(
        Patch(
            facecolor=publi_theo_pt_exp[0],
            edgecolor="none",
            alpha=0.5,
            label="Exp. closest local max. range"
        )
    )
    labels.append("Exp. closest local max. range")
    fig.legend(
        handles,
        labels,
        fontsize=legend_fontsize,
        # loc="lower center",
        # ncol=2,
        # bbox_to_anchor=(0.5, -0.12)
        ncol=1,
        loc="upper left",
        bbox_to_anchor=(0.92, 0.5)
    )
    
    
    if not publi_mode:
        plt.suptitle(f"{d_name} - {mode} - Histogram bins", fontsize=individual_title_fontsize)
    # # Add vertical line between plots
    # line = Line2D(
    #     [0.45, 0.45], [0.1, 1.0],
    #     transform=fig.transFigure,
    #     color='black',
    #     linewidth=1
    # )
    # fig.add_artist(line)
        
    plt.tight_layout()
    # Save svg figure
    plt.savefig(exp_root_path/exp_roots[d_name][0]/f"histogram_bins_{d_name.replace(' ', '')}_{score_name.replace(' ', '_')}_dropoutOrNot_nbins{n_bins}{svg_suffix}.svg", format='svg')
    plt.show()

# Exp 2 - calibration

In [ ]:
def compute_ece_by_bin(y_true, probs, n_bins=5, quantile_mode=False):
    """
    y_true: (N,)
    probs: (N, C)
    """
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)

    accuracies = (predictions == y_true).astype(float)

    if quantile_mode:
        bin_edges = np.quantile(confidences, np.linspace(0.0, 1.0, n_bins + 1))
    else:
        bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    bin_accs = []
    bin_confs = []
    bin_sizes = []
    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            mask = (confidences >= lower) & (confidences <= upper)
        else:
            mask = (confidences > lower) & (confidences <= upper)

        if np.any(mask):
            bin_accs.append(accuracies[mask].mean())
            bin_confs.append(confidences[mask].mean())
            bin_sizes.append(mask.sum() / len(y_true))
        else:
            bin_accs.append(np.nan)
            bin_confs.append(np.nan)
            bin_sizes.append(0.0)

    return bin_edges, bin_accs, bin_confs, bin_sizes

In [ ]:
# plot ECE with manual computation of ECE on same plot
# ECE and classwise ECE dicts
mode=TEST
publi_mode = True
n_bins = 10
quantile_mode = True
show_acc = True
show_mean_softmax = True
nr_rates=[0, 0.2, 0.4, 0.6]
for d_name in dataset_names:
    fig, axs = plt.subplots(1, len(included_losses[d_name]), figsize=(3 * len(included_losses[d_name]), 3), sharey=True)
    print(d_name, run_paths[d_name].keys())
    for i_l, l_name in enumerate(included_losses[d_name]):
        # Plot calibrated model line
        axs[i_l].plot([0, 1], [0, 1], "-.", color="gray", label="Calibrated model")
        for i_nr, nr in enumerate(nr_rates):
            if nr == 0.6 and d_name == "PTB":
                continue
            bin_conf_all, bin_acc_all, bin_sizes_all = [], [], []
            n_runs = len(predictions[d_name][l_name][nr])
            ece_bins_runs = []
            for i_run in range(n_runs):

                y_true = np.array(
                    predictions[d_name][l_name][nr][i_run]
                    [mode][target_names[d_name]][LABELS]
                )

                probs = np.array([
                    np.mean(pred, axis=1)
                    for pred in predictions[d_name][l_name][nr][i_run]
                    [mode][target_names[d_name]][PREDICTED]
                ])

                bin_edges, bin_accs, bin_confs, bin_sizes = compute_ece_by_bin(y_true, probs, n_bins=n_bins, quantile_mode=quantile_mode)
                bin_conf_all.append(bin_confs)
                bin_acc_all.append(bin_accs)
                bin_sizes_all.append(bin_sizes)
            print(d_name, l_name, nr, len(bin_conf_all), len(bin_acc_all))

            # Weighted average  across runs with bin size and ignoring NaN values
            w = np.array(bin_sizes_all)
            #print(w) # should be n_bins/100 if quantiles
            den = np.sum(np.where(np.isnan(np.array(bin_conf_all)), 0, w), axis=0)
            x_conf = np.array(bin_conf_all)
            num_conf = np.nansum(x_conf * w, axis=0)
            bin_conf_all = num_conf / den
            x_all = np.array(bin_acc_all)
            num_acc = np.nansum(x_all * w, axis=0)
            bin_acc_all = num_acc / den
            
            # Plot curve            
            if publi_mode:
                c_label = f"Noise rate {nr}"
            else:
                c_label = f"nr {nr}, test acc {test_acc:.3f}"
            axs[i_l].plot(bin_conf_all, bin_acc_all, 'x-', label=c_label, color=nr_colors[nr])
            axs[i_l].set_xlabel("Mean predicted probability")
            axs[i_l].set_title(f"{loss_viz_names[l_name]}")
            
            
            # Add test accuracies
            test_acc = avg_metrics[d_name][nr][l_name][mode]["acc " + target_names[d_name]]
            print(test_acc, std_metrics[d_name][nr][l_name][mode]["acc " + target_names[d_name]])
            if show_acc:
                axs[i_l].hlines(y = test_acc, xmin=0, xmax=0.1, color=nr_colors[nr], linestyle='--', label="Global mean")
            # Add mean accuracies
            if show_mean_softmax:
                mean_softmax = bin_conf_all.mean()
                axs[i_l].vlines(x = mean_softmax, ymin=0, ymax=0.1, color=nr_colors[nr], linestyle='--', label="Global mean" if not show_acc else None)
        
        axs[i_l].set_ylabel("Frac. of positives")
    axs[i_l].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    if not publi_mode:
        plt.suptitle(f"Calibration Curve for {d_name} - {mode}", fontsize=title_fontsize)
        
    # Add vertical line between plots
    line = Line2D(
        [0.43, 0.43], [0.1, 1.0],
        transform=fig.transFigure,
        color='black',
        linewidth=1
    )
    fig.add_artist(line)
    
    plt.tight_layout()
    
    # Save svg figure
    plt.savefig(
        exp_root_path/exp_roots[d_name][0]/f"calibration_curve_allNR_{d_name.replace(' ', '')}_{'quantile' if quantile_mode else 'uniform'}_dropoutOrNot_nbins{n_bins}{svg_suffix}.svg", 
        format='svg'
    )

    plt.show()
# 35s

# Show UQ MCDO

In [ ]:
mode = TEST

n_inference = 5
for i_uq, uq_n in enumerate(UQ_NAMES):
    for i_d, d_name in enumerate(dataset_names):
        fig, axs = plt.subplots(1, len(loss_names), figsize=(3 * len(loss_names), 2), sharey=True)
        target = target_names[d_name]
        loss_names = included_losses[d_name]
        for i_l, l_name in enumerate(loss_names):
            uq_list = []
            nr_rates_2 = sorted(list(predictions_MCDO[d_name][l_name].keys()))
            if d_name == "PTB":
                nr_rates_2.remove(0.6)
            for i_nr, nr in enumerate(nr_rates_2):
                uq_list.append([])
                for i_run in range(len(predictions_MCDO[d_name][l_name][nr])):
                    uq_list[i_nr] += predictions_MCDO[d_name][l_name][nr][i_run][mode][target][UQ]["mcdropout entropy"][uq_n]
            bp = axs[i_l].boxplot(
                uq_list,
                patch_artist=True)

            axs[i_l].set_xticks(np.arange(len(uq_list)) + 1, nr_rates_2)
            
            for i_e, patch in enumerate(bp['boxes']):
                patch.set_facecolor(nr_colors[nr_rates_2[i_e]])
            for j, line in enumerate(bp['medians']):
                line.set_color("black")
                line.set_linestyle("-")
            
            axs[i_l].set_xlabel("Noise rate", fontsize=10)
            if i_l == 0:
                axs[i_l].set_ylabel(f"{UQ_NAMES_SHORT.get(uq_n, uq_n)}", fontsize=10)
            axs[i_l].set_title(f"{d_name.replace('CNNTransf', '').replace('CNN', '')} - {loss_viz_names.get(l_name, l_name)}")
            # ax.set_ylim([-0.1, 1.1])
        # plt.suptitle(f"{UQ_NAMES_SHORT.get(uq_n, uq_n)} for {d_name} - {mode}")
        plt.tight_layout()
        
        # Save svg figure
        plt.savefig(
            exp_root_path/exp_roots[d_name][0]/f"{UQ_NAMES_SHORT.get(uq_n, uq_n)}_{d_name.replace(' ', '')}_dropoutOrNot_n_inf{n_inference}{svg_suffix}.svg", 
            format='svg'
        )
        
        plt.show()

In [ ]:
mode = TEST

n_inference = 5
for i_uq, uq_n in enumerate(UQ_NAMES):
    for i_d, d_name in enumerate(dataset_names):
        fig, axs = plt.subplots(1, len(loss_names), figsize=(3 * len(loss_names), 2), sharey=True)
        target = target_names[d_name]
        loss_names = included_losses[d_name]
        for i_l, l_name in enumerate(loss_names):
            uq_list = []
            nr_rates_2 = sorted(list(predictions_MCDO[d_name][l_name].keys()))
            if d_name == "PTB":
                nr_rates_2.remove(0.6)
            for i_nr, nr in enumerate(nr_rates_2):
                uq_list.append([])
                for i_run in range(len(predictions_MCDO[d_name][l_name][nr])):
                    uq_list[i_nr] += predictions_MCDO[d_name][l_name][nr][i_run][mode][target][UQ]["mcdropout entropy"][uq_n]

            sns.violinplot(uq_list, ax=axs[i_l], palette=[nr_colors[nr] for nr in nr_rates_2])

            axs[i_l].set_xticks(np.arange(len(uq_list)), nr_rates_2)
            
            for i_e, patch in enumerate(bp['boxes']):
                patch.set_facecolor(nr_colors[nr_rates_2[i_e]])
            for j, line in enumerate(bp['medians']):
                line.set_color("black")
                line.set_linestyle("-")
            
            axs[i_l].set_xlabel("Noise rate", fontsize=10)
            if i_l == 0:
                axs[i_l].set_ylabel(f"{UQ_NAMES_SHORT.get(uq_n, uq_n)}", fontsize=10)
            axs[i_l].set_title(f"{d_name.replace('CNNTransf', '').replace('CNN', '')} - {loss_viz_names.get(l_name, l_name)}")
            # ax.set_ylim([-0.1, 1.1])
        # plt.suptitle(f"{UQ_NAMES_SHORT.get(uq_n, uq_n)} for {d_name} - {mode}")
        plt.tight_layout()
        
        # Save svg figure
        plt.savefig(
            exp_root_path/exp_roots[d_name][0]/f"{UQ_NAMES_SHORT.get(uq_n, uq_n)}_{d_name.replace(' ', '')}_dropoutOrNot_n_inf{n_inference}_violin.svg", 
            format='svg'
        )
        
        plt.show()
#52s

In [ ]:
mode = TEST

n_inference = 5
n_bins = 40

for i_uq, uq_n in enumerate(UQ_NAMES):
    for i_d, d_name in enumerate(dataset_names):
        fig, axs = plt.subplots(4, len(loss_names), figsize=(3 * len(loss_names), 2 * 4), sharey=True)
        target = target_names[d_name]
        loss_names = included_losses[d_name]
        for i_l, l_name in enumerate(loss_names):
            uq_lst = []
            nr_rates_2 = sorted(list(predictions_MCDO[d_name][l_name].keys()))
            if d_name == "PTB":
                nr_rates_2.remove(0.6)
            for i_nr, nr in enumerate(nr_rates_2):
                for i_run in range(len(predictions_MCDO[d_name][l_name][nr])):
                    uq_list = predictions_MCDO[d_name][l_name][nr][i_run][mode][target][UQ]["mcdropout entropy"][uq_n]
                axs[i_nr, i_l].hist(
                    uq_list,
                    bins=n_bins,
                    color=nr_colors[nr]
                    )
                axs[i_nr, i_l].set_xlabel(UQ_NAMES_SHORT.get(uq_n, uq_n), fontsize=10)
                axs[i_nr, i_l].set_ylabel("Count", fontsize=10)
                
                if i_nr == 0:
                    axs[i_nr, i_l].set_title(f"{UQ_NAMES_SHORT.get(uq_n, uq_n)} - {loss_viz_names.get(l_name, l_name)}", fontsize=10)
            # axs[i_l].set_title(f"{d_name.replace('CNNTransf', '').replace('CNN', '')} - {loss_viz_names.get(l_name, l_name)}")
        plt.suptitle(f"{UQ_NAMES_SHORT.get(uq_n, uq_n)} for {d_name} - {mode}")
        plt.tight_layout()
        
        plt.show()

# 41s